# Optimizing Inventory Management on Supermarket Sales Data
### Dataset: Supermarket Sales (1000 records across 3 branches)


## 2.1 Data Cleaning

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import string

# Load the dataset
file_path = '/content/supermarket_sales - Sheet1.csv'
df = pd.read_csv(file_path)

# Show basic information about the dataset
df_info = df.info()
df_head = df.head()
print(df_head)

# Drop the 'gross margin percentage' column as it is constant and redundant
if 'gross margin percentage' in df.columns:
    df = df.drop(columns=['gross margin percentage'])

# Display the first few rows to confirm the column has been dropped
df.head()

# Checking the missing values
print(df.isnull().sum())

## 2.2 Data Conversion

In [ ]:
# 1. Convert 'Date' to datetime
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')
print(df['Date'])

# 2. Convert categorical columns to category type
df['Customer type'] = df['Customer type'].astype('category')
df['Gender'] = df['Gender'].astype('category')
df['Payment'] = df['Payment'].astype('category')
print(df['Customer type'])
print(df['Gender'])
print(df['Payment'])

# 3. Convert numerical columns if necessary
df['Total'] = pd.to_numeric(df['Total'], errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
print(df['Total'])
print(df['Quantity'])

# 4. Display the updated data types
print(df.dtypes)

## 2.3 Text Data Preprocessing

In [ ]:
# Select columns with text data
text_columns = ['Product line', 'Customer type', 'Payment']

# Define a function for text preprocessing
def preprocess_text(text):
    # Convert text to lowercase
    text = str(text).lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove leading/trailing whitespace
    text = text.strip()
    return text

# Apply text preprocessing to relevant columns
for col in text_columns:
    df[col] = df[col].apply(preprocess_text)
    print(df[col])

# Display the first few rows to verify preprocessing
print(df[text_columns].head())

## 2.4 Data Aggregation

In [ ]:
# Aggregating total sales and total quantity sold by City and Branch
city_sales = df.groupby('City').agg({'Total': 'sum', 'Quantity': 'sum'}).reset_index()
branch_sales = df.groupby('Branch').agg({'Total': 'sum', 'Quantity': 'sum'}).reset_index()
print(city_sales)
print(branch_sales)

# Aggregating by Product line
product_line_agg = df.groupby('Product line').agg({'Total': 'sum', 'Quantity': 'sum'}).reset_index()
print(product_line_agg)

# Summing total sales, taxes, and quantities
total_sales = df['Total'].sum()
total_taxes = df['Tax 5%'].sum()
total_quantity = df['Quantity'].sum()
print(f"Total Sales: {total_sales}")
print(f"Total Taxes: {total_taxes}")
print(f"Total Quantity Sold: {total_quantity}")

# Grouping by Customer type and Payment method
customer_type_agg = df.groupby('Customer type').agg({'Total': 'sum', 'Quantity': 'sum'}).reset_index()
payment_method_agg = df.groupby('Payment').agg({'Total': 'sum', 'Quantity': 'sum'}).reset_index()
print("Aggregation by Customer Type:")
print(customer_type_agg)
print("\nAggregation by Payment Method:")
print(payment_method_agg)

## 2.5 Data Splitting

In [ ]:
from sklearn.model_selection import train_test_split

# Reload dataset fresh for splitting
df_split = pd.read_csv('/content/supermarket_sales - Sheet1.csv')

# Use only numeric columns for features
numeric_cols = df_split.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [col for col in numeric_cols if col != 'Total']

X = df_split[numeric_cols]
y = df_split['Total']

# Split the data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")

## 2.6 Normalization and Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

df_scale = pd.read_csv('/content/supermarket_sales - Sheet1.csv')

# Select numerical columns for scaling
numerical_columns = ['Unit price', 'Quantity', 'Tax 5%', 'Total', 'gross income']

# 1. Min-Max Normalization (scaling values between 0 and 1)
min_max_scaler = MinMaxScaler()
df_min_max_scaled = df_scale.copy()
df_min_max_scaled[numerical_columns] = min_max_scaler.fit_transform(df_scale[numerical_columns])

# 2. Standardization (scaling to mean=0 and std=1)
standard_scaler = StandardScaler()
df_standard_scaled = df_scale.copy()
df_standard_scaled[numerical_columns] = standard_scaler.fit_transform(df_scale[numerical_columns])

print("Min-Max Normalized Data:")
print(df_min_max_scaled[numerical_columns].head())
print("\nStandardized Data:")
print(df_standard_scaled[numerical_columns].head())

## 3.1 Sales by City and Branch

In [ ]:
df = pd.read_csv('/content/supermarket_sales - Sheet1.csv')

# Grouping the data by City and Branch
city_branch_sales = df.groupby(['City', 'Branch'])['Total'].sum().reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(x='City', y='Total', hue='Branch', data=city_branch_sales, palette='Set2')
plt.title('Total Sales by City and Branch', fontsize=16)
plt.xlabel('City', fontsize=12)
plt.ylabel('Total Sales (USD)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3.2 Product Line Performance

In [ ]:
# Grouping the data by Product line
product_performance = df.groupby('Product line')['Total'].sum().reset_index()
product_performance = product_performance.sort_values(by='Total', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Total', y='Product line', data=product_performance, palette='coolwarm')
plt.title('Total Sales by Product Line', fontsize=16)
plt.xlabel('Total Sales (USD)', fontsize=12)
plt.ylabel('Product Line', fontsize=12)
plt.tight_layout()
plt.show()

## 3.3 Sales Trends Over Time

In [ ]:
# Convert the 'Date' column to datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Grouping the data by Date
sales_trend = df.groupby('Date')['Total'].sum().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(x='Date', y='Total', data=sales_trend, marker='o', color='b')
plt.title('Sales Trends Over Time', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Total Sales (USD)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3.4 Customer Behaviour

In [ ]:
# Sales by Gender
gender_sales = df.groupby('Gender')['Total'].sum().reset_index()

# Sales by Payment Method
payment_sales = df.groupby('Payment')['Total'].sum().reset_index()

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
sns.barplot(x='Gender', y='Total', data=gender_sales, palette='Set2')
plt.title('Total Sales by Gender')
plt.xlabel('Gender')
plt.ylabel('Total Sales (USD)')

plt.subplot(1, 2, 2)
sns.barplot(x='Payment', y='Total', data=payment_sales, palette='Set3')
plt.title('Total Sales by Payment Method')
plt.xlabel('Payment Method')
plt.ylabel('Total Sales (USD)')

plt.tight_layout()
plt.show()

## 4.1 Pie Chart - Payment Methods Distribution

In [ ]:
df = pd.read_csv('/content/supermarket_sales - Sheet1.csv')

# Calculate the counts of each payment method
payment_method_counts = df['Payment'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(payment_method_counts, labels=payment_method_counts.index,
        autopct='%1.1f%%', colors=sns.color_palette("Set2"))
plt.title('Distribution of Payment Methods')
plt.tight_layout()
plt.show()

## 4.2 Scatter Plot - Rating vs Gross Income

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x='Rating', y='gross income', data=df, hue='Branch', palette='coolwarm')
plt.title('Correlation between Rating and Gross Income')
plt.xlabel('Customer Rating')
plt.ylabel('Gross Income')
plt.tight_layout()
plt.show()

## 4.3 Bar Plot - Average Rating by Product Line

In [ ]:
# Grouping the data by Product line and calculating the average rating
avg_rating_by_product_line = df.groupby('Product line')['Rating'].mean().sort_values()

plt.figure(figsize=(10, 6))
sns.barplot(x=avg_rating_by_product_line.index, y=avg_rating_by_product_line.values, palette="Blues_d")
plt.title('Average Customer Rating by Product Line')
plt.xlabel('Product Line')
plt.ylabel('Average Rating')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()